In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize, integrate
from helper import *
import seaborn as sns
import pandas as pd
from scipy.spatial.transform import Rotation as R
sns.set_style("darkgrid")

In [ ]:
import spiceypy as spice
spice.furnsh(r"spice_data\de442s.bsp")
spice.furnsh(r"spice_data\mar099.bsp")
spice.furnsh(r"spice_data\naif0012.tls")

In [ ]:
mu_sun= 1.327e11 ## km^3/s^2  
mu_earth= 3.986e5 ## km^3/s^2  # 3.986e5
mu_mars= 4.2830e4 ## km^3/s^2
mu_venus = 32.4776e4

mu_moon = 0.4902e4 ## km^3/s^2 Not given in paper

SOI_earth = 923502.24 # km
SOI_mars = 577723.87 # km 577723.87 
SOI_moon = 66300

D_mars = 2.279e8
D_earth = 1.496e8
D_venus = 1.0815e8
D_moon = 384400 # km

r_LEO = 463
R_LEO = 6378.2 + r_LEO

w_earth = 1.99177621e-7 # rad/s
w_mars = 1.05850987e-7 # rad/s
w_venus = 3.23861161e-7
w_moon = 2.6653e-6 # rad/s

r_LMO = 200
R_LMO = 3397.0 + r_LMO

In [ ]:
dt = 30
time_s, time_e = 0, (60*60*24*365.25)
ts = int(time_e/(dt))
print(ts)
t_points = np.linspace(time_s, time_e, ts)


In [ ]:
def angle_vector_plane(vector):
    normal = [0,0,1]
    v_unit = vector / np.linalg.norm(vector)
    n_unit = normal / np.linalg.norm(normal)
    
    dot_product = np.dot(v_unit, n_unit)
    alpha = np.arccos(dot_product)
    
    theta_rad = (np.pi / 2) - alpha
    return theta_rad

# Example: Plane is the XY plane (normal is Z-axis [0,0,1])
# Vector is at 45 degrees to the XY plane [1, 0, 1]
vec = np.array([10, 0, 1])


print(f"Angle: {angle_vector_plane(vec):.2f} rad")

def perpendicular_norm( a ):
    perp = np.cross([0,0,1], a ) ## FUCKS SAKE
    return perp/np.linalg.norm(perp)

def map_to_2D(a):
    perp_unit = perpendicular_norm(a)
    r = R.from_rotvec(angle_vector_plane(a) * perp_unit )
    # print(a)
    # print(perp_unit)
    # print(angle_vector_plane(a))
    earth_pos_rot = r.apply(a)
    # print(earth_pos_rot)
    return earth_pos_rot



In [ ]:
earth_pos = earth[:,:3]
earth_2D = []
for i in range(len(earth_pos)):
    earth_2D.append(map_to_2D(earth_pos[i])) 
    # print(earth_pos[i,:])
earth_2D = np.array(earth_2D)
earth_2Dx = earth_2D[:,0]
earth_2Dy = earth_2D[:,1]
earth_2Dz = earth_2D[:,2]


mars_pos = mars[:,:3]
mars_2D = []
for i in range(len(mars_pos)):
    mars_2D.append(map_to_2D(mars_pos[i])) 
    # print(earth_pos[i,:])
mars_2D = np.array(mars_2D)
mars_2Dx = mars_2D[:,0]
mars_2Dy = mars_2D[:,1]
mars_2Dz = mars_2D[:,2]

venus_pos = venus[:,:3]
venus_2D = []
for i in range(len(venus_pos)):
    venus_2D.append(map_to_2D(venus_pos[i])) 
    # print(earth_pos[i,:])
venus_2D = np.array(venus_2D)
venus_2Dx = venus_2D[:,0]
venus_2Dy = venus_2D[:,1]
venus_2Dz = venus_2D[:,2]

venus_2Dx

In [ ]:
def find_mars_diff(dt):
    dt = 30
    time_s, time_e = 0, (60*60*24*687)
    ts = int(time_e/(dt))
    print(ts)
    t_points = np.linspace(time_s, time_e, ts)

    FRAME = "ECLIPJ2000" # LOAD IN EPHEMERIS DATA
    OBSERVER = "SUN"
    mars = spice.spkezr("MARS",t_points,FRAME,"NONE", OBSERVER)[0]
    mars_pos = mars[:,:3]
    mars_2D = []
    for i in range(len(mars_pos)):
        mars_2D.append(map_to_2D(mars_pos[i])) 
        # print(earth_pos[i,:])
    mars_2D = np.array(mars_2D)

    init_phase_mars = np.arccos(mars_2D[0,0]/D_mars) 

    for t in t_points:
        rx_mars = D_mars*np.cos(w_mars*t + init_phase_mars)
        ry_mars = D_mars*np.sin(w_mars*t + init_phase_mars)
        r_mars.append([rx_mars,ry_mars,0])

    deviation_m = []
    r_mars = []
    for i in range(len(mars_pos)):
        deviation_m.append(np.linalg.norm(mars_2D[i,:] - r_mars[i,:]) )
        print(np.linalg.norm(mars_2D[i,:] - r_mars[i,:]))

